# CASMI26: long fingerprint-model run (~11 h)

Fine-tunes the fold-safe FPNet on real + simulated spectra with the hardest same-formula-window negatives
(16 of 63 per query). Every hour the latest checkpoint goes to the private Kaggle dataset
`giantwizard/casmi26-jc-fplong` so it can be scored while training runs.

**Run all, keep this tab open and the computer awake.** If it disconnects, Run all again: it resumes from the last
hourly checkpoint on Drive (`MyDrive/casmi26/fplong/`).
Needs: GPU runtime (L4), the `KAGGLE_API_TOKEN` secret, the Google Drive prompt.


In [ ]:
# 1. Drive, credentials, dependencies, inputs
import os, sys, json, glob, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
def sh(cmd, check=True):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-2000:], r.stderr[-2000:], flush=True)
    if check and r.returncode: raise RuntimeError(cmd)
    return r
sh(f'{sys.executable} -m pip install -q -U kaggle rdkit numba pyarrow')
SIMD = '/content/drive/MyDrive/casmi26/fpsim_full'
OUT = '/content/drive/MyDrive/casmi26/fplong'; os.makedirs(OUT, exist_ok=True)
print('simulated parts on Drive:', len(glob.glob(SIMD + '/*_meta.parquet')))
R = '/content/fp'; os.makedirs(R, exist_ok=True)
def get(ds, dst, f=None):
    if os.path.exists(dst + '/.ok'): return
    sh(f'kaggle datasets download {ds} -p {dst} --unzip' + (f' -f {f}' if f else ''))
    open(dst + '/.ok', 'w').close()
def get_files(ds, dst):
    if os.path.exists(dst + '/.ok'): return
    os.makedirs(dst, exist_ok=True)
    names = [l.split(',')[0] for l in sh(f'kaggle datasets files {ds} --csv --page-size 500').stdout.strip().splitlines()[1:]]
    for n in names:
        if not n.startswith('pilot_part') and not os.path.exists(f'{dst}/{n}'):
            sh(f'kaggle datasets download {ds} -f {n} -p {dst} -q')
            if os.path.exists(f'{dst}/{n}.zip'):
                sh(f'cd {dst} && unzip -oq {n}.zip && rm {n}.zip')
    open(dst + '/.ok', 'w').close()
get('giantwizard/casmi26-jc-fpsim', R + '/in')
get_files('giantwizard/casmi26-jc-fpsim-sim', R + '/in2')
get('ahmedberatozer/casmi26-v2-pool', R + '/pool')
get('ahmedberatozer/casmi26-v4b-models', R + '/v4b')
get('dmitriigluzdov/casmi26-fold-safe-fpnet', R + '/fs', 'fpnet_fold0.pt')
CODE = os.path.dirname(os.path.dirname(glob.glob(R + '/v4b/**/casmi/fpnet.py', recursive=True)[0]))
W = R + '/run/work'; os.makedirs(W, exist_ok=True)
for f in glob.glob(R + '/pool/**/*', recursive=True):
    if os.path.isfile(f) and not os.path.exists(W + '/' + os.path.basename(f)):
        os.symlink(f, W + '/' + os.path.basename(f))
for f in ('spec_off.npy', 'spec_mz.npy', 'spec_it.npy', 'spec_meta.parquet', 'split.parquet'):
    if not os.path.exists(W + '/' + f):
        os.symlink(glob.glob(R + '/in/**/' + f, recursive=True)[0], W + '/' + f)
IN = R + '/in2'
CK0 = glob.glob(R + '/fs/**/fpnet_fold0.pt', recursive=True)[0]


In [ ]:
# 2. training script (embedded copy of bench/fpsim/train_sim.py)
TS = '/content/train_sim.py'
open(TS, 'w').write('"""Fine-tune an FPNet checkpoint on real training spectra plus GLACIER-simulated spectra of pool structures\n(MIST/FRIGID-style forward-model augmentation), with Gluzdov\'s mass-window ranking loss. sim_p = 0 is the control.\n\nReal spectra: casmi traindata.Data with the fold-safe holdout (hold, np) excluded. Simulated spectra map straight to\ntheir pool entry (pid), so structures with no measured spectrum (COCONUT) become training targets.\n\nusage: train_sim.py --init fpnet_fold0.pt --sim sim/pilot --sim_p 0.5 --steps 3000 --tag NAME [--device mps]\n"""\nimport argparse, glob, json, math, os, sys, time\nimport numpy as np, pandas as pd\n\nap = argparse.ArgumentParser()\nap.add_argument(\'--code\', default=\'/Users/b/casmi26/inputs/v4b/code\')\nap.add_argument(\'--run\', default=\'/Users/b/casmi26/fp/run\')\nap.add_argument(\'--init\', default=\'\', help=\'checkpoint to fine-tune; empty = new model of --arch\')\nap.add_argument(\'--arch\', default=\'512,8,8\', help=\'d,layers,heads for a new model\')\nap.add_argument(\'--hard_k\', type=int, default=0, help=\'replace this many window negatives per query with the hardest ones under the current model (mined from 4x as many window draws)\')\nap.add_argument(\'--save_every\', type=float, default=1.0, help=\'hours between intermediate checkpoints (Colab disconnect safety)\')\nap.add_argument(\'--sim\', default=\'\')\nap.add_argument(\'--sim_p\', type=float, default=0.5)\nap.add_argument(\'--exclude_sim_pids\', default=\'\', help=\'npy of pids whose simulated spectra are dropped (ablation)\')\nap.add_argument(\'--steps\', type=int, default=3000)\nap.add_argument(\'--bs\', type=int, default=128)\nap.add_argument(\'--lr\', type=float, default=1e-4)\nap.add_argument(\'--warmup\', type=int, default=200)\nap.add_argument(\'--merge_p\', type=float, default=0.3)\nap.add_argument(\'--w_bce\', type=float, default=1.0)\nap.add_argument(\'--w_el\', type=float, default=0.1)\nap.add_argument(\'--device\', default=\'mps\')\nap.add_argument(\'--seed\', type=int, default=0)\nap.add_argument(\'--tag\', required=True)\nap.add_argument(\'--outdir\', default=\'/Users/b/casmi26/fp\')\nap.add_argument(\'--hours\', type=float, default=0.0, help=\'stop early after this many hours (0 = steps only)\')\nap.add_argument(\'--holdout\', default=\'hold,np\', help="folds kept out of training (\'\' = train on everything, deployment)")\na = ap.parse_args()\nsys.path.insert(0, a.code)\nos.chdir(a.run)\nimport torch  # noqa: E402\nimport torch.nn.functional as F  # noqa: E402\nfrom casmi import fpnet, traindata  # noqa: E402\n\nD = traindata.Data(\'cpu\', seed=a.seed, holdout_folds=tuple(f for f in a.holdout.split(\',\') if f))\nrng = np.random.default_rng(a.seed + 7)\nD.rng = rng\n\n# ---- simulated spectra ------------------------------------------------------------------------------------------\nS = None\nif a.sim and a.sim_p > 0:\n    parts = sorted(f for pre in a.sim.split(\',\') for f in glob.glob(pre + \'_part*_meta.parquet\'))\n    metas, mzs, its, offs = [], [], [], []\n    base = 0\n    for mp in parts:\n        pre = mp[:-len(\'_meta.parquet\')]\n        m = pd.read_parquet(mp); o = np.load(pre + \'_off.npy\')\n        metas.append(m); mzs.append(np.load(pre + \'_mz.npy\')); its.append(np.load(pre + \'_it.npy\'))\n        offs.append(o[:-1] + base); base += int(o[-1])\n    meta = pd.concat(metas, ignore_index=True)\n    ADD = {\'[M+H]+\': (0, 1.007276), \'[M+Na]+\': (4, 22.989218)}       # adduct_ix as in spec_meta, ion mass shift\n    S = dict(add_ix=np.array([ADD[x][0] for x in meta.adduct.values], np.int64),\n             add_mass=np.array([ADD[x][1] for x in meta.adduct.values], np.float64),\n             pid=meta.pid.values.astype(np.int64), ce=meta.ce.values.astype(np.float64),\n             off=np.concatenate(offs + [np.array([base])]), mz=np.concatenate(mzs), it=np.concatenate(its))\n    keep = np.ones(len(S[\'pid\']), bool)\n    if a.exclude_sim_pids:\n        keep &= ~np.isin(S[\'pid\'], np.load(a.exclude_sim_pids))\n    S[\'idx\'] = np.where(keep)[0]\n    print(f\'simulated spectra {len(S["pid"]):,} (usable {len(S["idx"]):,}) over {len(np.unique(S["pid"])):,} structures\', flush=True)\n# precursor m/z of [M+H]+ from the pool\'s neutral mass\nPROTON = 1.007276\n\n\ndef sim_item(j, aug=True):\n    s, e = S[\'off\'][j], S[\'off\'][j + 1]\n    mz = S[\'mz\'][s:e].astype(np.float64); it = S[\'it\'][s:e].astype(np.float64)\n    prec = float(D.pmass[S[\'pid\'][j]] + S[\'add_mass\'][j])\n    if aug:\n        if len(mz) > 4:\n            keep = rng.random(len(mz)) > rng.uniform(0.0, 0.3)\n            if keep.sum() >= 3:\n                mz, it = mz[keep], it[keep]\n        it = it * np.exp(rng.normal(0.0, 0.3, size=len(it)))\n        mz = mz * (1.0 + rng.normal(0.0, 4e-6, size=len(mz)))\n    pm, pi = fpnet.prep_peaks(mz, it, prec)\n    return dict(mz=pm, it=pi, prec=prec, adduct_ix=int(S[\'add_ix\'][j]), ce=float(S[\'ce\'][j]), ce_known=1.0, n_merged=1, mode=1.0)\n\n\ndef batch():\n    n_sim = int(rng.binomial(a.bs, a.sim_p)) if S is not None else 0\n    ids = D.sample_ids(a.bs - n_sim)\n    items = [D.item(int(i), a.merge_p, True) for i in ids]\n    pos = list(D.sid2pid[D.sid[ids]])\n    if n_sim:\n        js = rng.choice(S[\'idx\'], n_sim, replace=False)\n        items += [sim_item(int(j)) for j in js]\n        pos += list(S[\'pid\'][js])\n    pos = np.array(pos, np.int64)\n    return fpnet.collate(items, dev), pos, D.negatives(pos)\n\n\ndev = a.device\ntorch.manual_seed(a.seed)\nPFP = D.PFP.to(dev); ELEM = D.elem.to(dev)\nif a.init:\n    model = fpnet.load_model(a.init, dev).train()\nelse:\n    _d, _l, _h = (int(x) for x in a.arch.split(\',\'))\n    model = fpnet.FPNet(D.nbits, d=_d, layers=_l, heads=_h).to(dev).train()\n    print(f\'new FPNet d {_d} layers {_l} heads {_h}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params\', flush=True)\ndecay = [p for n, p in model.named_parameters() if p.ndim >= 2]; no_decay = [p for n, p in model.named_parameters() if p.ndim < 2]\nopt = torch.optim.AdamW([dict(params=decay, weight_decay=0.01), dict(params=no_decay, weight_decay=0.0)], lr=a.lr, betas=(0.9, 0.98))\nt0 = time.time(); ema = {}\ntarget = torch.zeros(a.bs, dtype=torch.long, device=dev)\nfor step in range(1, a.steps + 1):\n    if a.hours and time.time() - t0 > a.hours * 3600:\n        print(\'time budget reached at step\', step, flush=True); break\n    prog = step / a.steps\n    lr = a.lr * min(1.0, step / a.warmup) * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * prog)))\n    for g in opt.param_groups:\n        g[\'lr\'] = lr\n    inp, pos, negs = batch()\n    if a.hard_k:                                   # hardest same-window negatives under the current model\n        pool_negs = np.concatenate([negs] + [D.negatives(pos) for _ in range(3)], 1)\n        with torch.no_grad():\n            if dev == \'cuda\':\n                with torch.autocast(\'cuda\', dtype=fpnet.autocast_dtype(\'cuda\')):\n                    z0, _ = model(*inp)\n            else:\n                z0, _ = model(*inp)\n            pc = torch.as_tensor(pool_negs, device=dev)\n            sc0 = torch.einsum(\'bkn,bn->bk\', fpnet.unpack_gpu(PFP[pc], D.nbits), z0.float())\n            sc0[pc == torch.as_tensor(pos, device=dev)[:, None]] = -1e9\n            hard = torch.gather(pc, 1, sc0.topk(a.hard_k, 1).indices).cpu().numpy()\n        negs = np.concatenate([hard, negs[:, :negs.shape[1] - a.hard_k]], 1)\n    cand = torch.as_tensor(np.concatenate([pos[:, None], negs], 1), device=dev)\n    if dev == \'cuda\':\n        with torch.autocast(\'cuda\', dtype=fpnet.autocast_dtype(\'cuda\')):\n            z, el = model(*inp)\n        z = z.float(); el = el.float()\n    else:\n        z, el = model(*inp)\n    fp = fpnet.unpack_gpu(PFP[cand], D.nbits)\n    sc = torch.einsum(\'bkn,bn->bk\', fp, z)\n    l_rank = F.cross_entropy(sc, target)\n    l_bce = F.binary_cross_entropy_with_logits(z, fp[:, 0])\n    l_el = F.smooth_l1_loss(el, ELEM[torch.as_tensor(pos, device=dev)])\n    loss = l_rank + a.w_bce * l_bce + a.w_el * l_el\n    opt.zero_grad(set_to_none=True)\n    loss.backward()\n    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n    opt.step()\n    vals = dict(loss=loss.item(), rank=l_rank.item(), acc=(sc.argmax(1) == 0).float().mean().item())\n    for k, v in vals.items():\n        ema[k] = v if step == 1 else 0.98 * ema[k] + 0.02 * v\n    if a.save_every and time.time() - getattr(model, \'_last_save\', t0) > a.save_every * 3600:\n        model.eval(); fpnet.save_ckpt(os.path.join(a.outdir, f\'{a.tag}_partial.pt\'), model, step, extra=dict(args=vars(a))); model.train()\n        model._last_save = time.time(); print(\'saved partial at step\', step, flush=True)\n    if step % 100 == 0:\n        print(f\'step {step} lr {lr:.2e} \' + \' \'.join(f\'{k} {v:.4f}\' for k, v in ema.items()) +\n              f\' {step / (time.time() - t0):.2f} it/s\', flush=True)\nmodel.eval()\nout = os.path.join(a.outdir, f\'{a.tag}.pt\')\nfpnet.save_ckpt(out, model, step, extra=dict(args=vars(a)))\nprint(\'saved\', out, f\'{(time.time() - t0) / 60:.1f} min\', flush=True)\n')
print('wrote', TS)


In [ ]:
# 3. start training in the background (resumes from the last partial checkpoint if there is one)
import time
HOURS, HARD_K, TAG = 11.0, 16, 'fplong'
SIM = f'{SIMD}/mh,{SIMD}/mna'
part = f'{OUT}/{TAG}_partial.pt'
init = part if os.path.exists(part) else CK0
done_h = 0.0
if os.path.exists(f'{OUT}/hours_done.txt'):
    done_h = float(open(f'{OUT}/hours_done.txt').read() or 0)
left = max(0.5, HOURS - done_h)
print('init from', init, '| hours left', round(left, 2))
cmd = (f'{sys.executable} {TS} --code {CODE} --run {R}/run --init {init} --device cuda --bs 256 --lr 1e-4 '
       f'--steps 65000 --hours {left} --sim {SIM} --sim_p 0.5 --hard_k {HARD_K} --save_every 1 '
       f'--tag {TAG} --outdir {OUT} >> {OUT}/train_{TAG}.log 2>&1')
proc = subprocess.Popen(cmd, shell=True)
T_START = time.time()
print('training started, pid', proc.pid)


In [ ]:
# 4. every hour: publish the latest checkpoint + log to the private Kaggle dataset; ends when training ends
import shutil
P = '/content/pub_long'; os.makedirs(P, exist_ok=True)
json.dump({'title': 'casmi26-jc-fplong', 'id': 'giantwizard/casmi26-jc-fplong', 'licenses': [{'name': 'other'}]},
          open(P + '/dataset-metadata.json', 'w'))
def publish(note):
    for f in (f'{OUT}/{TAG}_partial.pt', f'{OUT}/{TAG}.pt', f'{OUT}/train_{TAG}.log'):
        if os.path.exists(f):
            shutil.copy(f, P)
    if sh(f'kaggle datasets create -p {P} -q', check=False).returncode:
        sh(f'kaggle datasets version -p {P} -m "{note}" -q', check=False)
last = 0
while proc.poll() is None:
    time.sleep(60)
    open(f'{OUT}/hours_done.txt', 'w').write(str(done_h + (time.time() - T_START) / 3600))
    if time.time() - last > 3600 and os.path.exists(f'{OUT}/{TAG}_partial.pt'):
        publish(f'partial {(time.time() - T_START) / 3600:.1f} h'); last = time.time()
        print(open(f'{OUT}/train_{TAG}.log').read()[-400:], flush=True)
publish('final')
print(open(f'{OUT}/train_{TAG}.log').read()[-1500:])
print('final checkpoint:', os.path.exists(f'{OUT}/{TAG}.pt'))
